# U09_C1 — Protein Properties and Motifs
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 9:** Predictive Methods using Protein Sequence
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L23–L24, Practical S10 (ProtParam & PROSITE)
**Before this:** U08_C1

## 1. What you will learn
- **Compute** fundamental physicochemical properties (molecular weight, pI, GRAVY hydropathy) using Biopython's `Bio.SeqUtils.ProtParam` [LO6].
- **Generate** Kyte–Doolittle hydropathy plots to contrast soluble globular proteins with hydrophobic transmembrane helices [LO6].
- **Translate** PROSITE regular expression syntax (`[LIVM]`, `x(2,4)`, `{C}`) into standard Python regex [LO6].
- **Quantify** how a single pathogenic amino acid substitution (HbS Glu6Val) shifts molecular weight, isoelectric point, and surface hydrophobicity [LO6].


## 2. Biological question
Sickle-cell hemoglobin (HbS) polymerizes into rigid fibers under low oxygen tension, causing erythrocyte sickling and vaso-occlusive crises. Biochemically, why does substituting a single residue (glutamic acid to valine at position 6) produce such catastrophic polymerization? How do computational sequence analysis tools quantify the change in molecular weight, net charge (pI), and hydrophobicity?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U09_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: Physicochemical Properties with ProtParam
### Theory
Expasy ProtParam computes physical and chemical parameters based on primary sequence:
1. **Molecular Weight ($M_r$)**: Sum of isotopic average masses of amino acids minus $(N-1) \times 18.015$ Da (water lost in peptide bond formation).
2. **Isoelectric Point (pI)**: The pH at which the protein carries net zero electrical charge. Acidic residues (Asp, Glu) lower the pI; basic residues (Lys, Arg, His) raise the pI.
3. **GRAVY (Grand Average of Hydropathy)**: Average of Kyte–Doolittle hydropathy values across all residues. Negative values indicate hydrophilic/globular proteins; positive values indicate hydrophobic/transmembrane proteins.


In [ ]:
# Worked Example: Comparing Normal HbA vs Sickle-Cell HbS with ProtParam
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

hbb_seq = str(SeqIO.read(DATA_DIR / "sequences" / "hbb_human_protein.fasta", "fasta").seq)
hbs_seq = str(SeqIO.read(DATA_DIR / "sequences" / "hbs_human_protein.fasta", "fasta").seq)

pa_hbb = ProteinAnalysis(hbb_seq)
pa_hbs = ProteinAnalysis(hbs_seq)

print("=== Normal Adult Hemoglobin Beta (HbA) ===")
print(f"Length:           {len(hbb_seq)} amino acids")
print(f"Molecular Weight: {pa_hbb.molecular_weight():.2f} Da")
print(f"Isoelectric (pI): {pa_hbb.isoelectric_point():.2f}")
print(f"GRAVY Score:      {pa_hbb.gravy():.3f} (Globular/Soluble)")

print("\n=== Sickle-Cell Hemoglobin (HbS Glu6Val) ===")
print(f"Length:           {len(hbs_seq)} amino acids")
print(f"Molecular Weight: {pa_hbs.molecular_weight():.2f} Da")
print(f"Isoelectric (pI): {pa_hbs.isoelectric_point():.2f}")
print(f"GRAVY Score:      {pa_hbs.gravy():.3f} (Shifted more hydrophobic)")

mw_diff = pa_hbs.molecular_weight() - pa_hbb.molecular_weight()
pi_diff = pa_hbs.isoelectric_point() - pa_hbb.isoelectric_point()
print(f"\nNet effect of Glu6Val: MW shift = {mw_diff:.2f} Da | pI shift = {pi_diff:+.2f} pH units")


## 4. Concept 2: Kyte–Doolittle Hydropathy Profiles
### Theory
Kyte and Doolittle (1982) assigned an empirical hydropathy value to each of the 20 amino acids (from Isoleucine $+4.5$ to Arginine $-4.5$).
A sliding window computes local hydrophobicity along the peptide chain:
- Peaks $> +1.6$ (over a 19–21 residue window) indicate **transmembrane alpha-helices**.
- Mostly negative regions ($< 0$) indicate **water-exposed, hydrophilic loops**.


In [ ]:
# Worked Example: Plotting Kyte-Doolittle Hydropathy Profile for HBB
import matplotlib.pyplot as plt
from Bio.SeqUtils.ProtParam import ProtParamData

window_size = 9
# ProteinAnalysis.protein_scale returns a list of window averages
kd_scores = pa_hbb.protein_scale(ProtParamData.kd, window=window_size)
positions = list(range(window_size // 2, window_size // 2 + len(kd_scores)))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(positions, kd_scores, color="navy", lw=1.5)
ax.axhline(0, color="gray", linestyle="--", alpha=0.7)
ax.axhline(1.6, color="red", linestyle=":", alpha=0.7, label="Transmembrane threshold (>1.6)")
ax.set_xlabel("Amino Acid Position")
ax.set_ylabel("Hydropathy Score (Kyte-Doolittle)")
ax.set_title("HBB Hydropathy Profile (Soluble Globular Protein)")
ax.set_ylim(-3.5, 3.5)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(OUT_DIR / "hbb_hydropathy.png", dpi=100)
plt.show()


## 5. Concept 3: PROSITE Motifs and Regular Expressions
### Theory
PROSITE defines biological sequence motifs using a standard pattern grammar:
- `[ALT]`: Any of Ala, Leu, or Thr.
- `{P}`: Any amino acid *except* Proline.
- `x`: Any amino acid.
- `x(3)`: Exactly 3 of any amino acid (`.{3}` in regex).
- `x(2,4)`: Between 2 and 4 of any amino acid (`.{2,4}` in regex).
- `<` and `>`: Sequence N-terminus and C-terminus.

**Globin Family Signature (PS00017):**
`[LIVMFY]-[A-Z]-H-[A-Z]-[A-Z]-[LIVMFY]` matches the proximal histidine region.


In [ ]:
# Worked Example: Scanning for PROSITE motifs using Python re
import re

# PROSITE globin distal histidine pattern: [LIVMF]-x(2)-H-x(2)-[KR]
# Convert to Python regular expression:
regex_pattern = r"[LIVMF].{2}H.{2}[KR]"

matches = list(re.finditer(regex_pattern, hbb_seq))
print(f"Scanning HBB for globin motif pattern '{regex_pattern}':")
for m in matches:
    print(f"  Match at 0-based indices {m.start()}:{m.end()} (1-based {m.start()+1}..{m.end()}): '{m.group()}'")


## 6. Summary
- `Bio.SeqUtils.ProtParam.ProteinAnalysis` calculates molecular weight, pI, and GRAVY hydropathy.
- The sickle-cell mutation (E6V) eliminates an acidic charge, shifting the pI from 6.74 to 7.13 and increasing hydrophobicity.
- Kyte–Doolittle hydropathy profiles identify transmembrane helices and surface loops.
- PROSITE patterns map directly to Python regular expressions (`[LIVM]`, `.{N}`).


## 7. Next Steps
- Learn how secondary structure is predicted and scored with $Q_3$ in `U09_C2_Secondary_Structure_Prediction_Scoring.ipynb`.
- Solve protein property and motif exercises in `U09_P_Protein_Prediction.ipynb`.
